In [6]:
import os
import pandas as pd
from dataclasses import dataclass
from notebooks._utils import calculate_accuracy

data_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
dataset_names = ["hotpot-reasoning-debug"]

@dataclass
class Args:
    num_paraphrases: int
    repeat: int
    method: str = "per_prompt"
    temperature: float = 1.0
    top_p: float = 0.95
    max_new_tokens: int = 2048
    reasoning: bool = True
    num_fewshots: int = 0
    additional_paraphrases_file: str = None

@dataclass
class DummyDataset:
    dataset_root:str

In [20]:
from utils import get_baseline_dumpfile

        
# for model_name in ["qwen3_1.7b", "qwen3_4b", "qwen3_8b", "qwen3_14b", "qwen3_32b"]:
for model_name in ["qwen3_14b"]:
    for dataset_name in dataset_names:
        dataset_path = os.path.join(data_root, "hotpot-reasoning-debug", model_name)
        dataset = DummyDataset(dataset_root=dataset_path)
        
        baseline_args = Args(num_paraphrases=0, repeat=4)
        paraphrase_args = Args(num_paraphrases=1, repeat=2)

        baseline_filename = get_baseline_dumpfile(dataset, baseline_args)
        
        print(f"Dataset: {dataset_name.split('-')[0]}, Model: {model_name}")
        print(f"=================== {baseline_args.num_paraphrases + 1} prompt x {baseline_args.repeat} thinkings =====================")
        if os.path.exists(baseline_filename) is False:
            print(f"Baseline file {baseline_filename} does not exist.")
        else:
            baseline_df = pd.read_feather(baseline_filename)
            baseline_df = baseline_df[baseline_df['thinking'].str.len() > 0]
            baseline_acc = calculate_accuracy(
                baseline_df, is_multichoice=False,
                birdirect=True, use_generation=True, by_probs=False,
                label=f"Average accuracy")

        print(f"=================== {paraphrase_args.num_paraphrases + 1} prompt x {paraphrase_args.repeat} thinkings =====================")
        paraphrase_filename = get_baseline_dumpfile(dataset, paraphrase_args)
        if os.path.exists(paraphrase_filename) is False:
            print(f"Paraphrase file {paraphrase_filename} does not exist.")
        else:
            paraphrase_df = pd.read_feather(paraphrase_filename)
            paraphrase_df = paraphrase_df[paraphrase_df['thinking'].str.len() > 0]
            paraphrase_acc = calculate_accuracy(
                paraphrase_df, is_multichoice=False,
                birdirect=True, use_generation=True, by_probs=False,
                label=f"Average accuracy")


Dataset: hotpot, Model: qwen3_14b
=================== 1 prompt x 4 thinkings =====================
Acc: 0.3856 ==> 🏷️ Average accuracy
=================== 2 prompt x 2 thinkings =====================
Acc: 0.3965 ==> 🏷️ Average accuracy


In [10]:
uuids = baseline_df["uuid"].unique().tolist()

In [22]:
for uuid in uuids:
    baseline = baseline_df[baseline_df["uuid"] == uuid].reset_index(drop=True)
    paraphrase = paraphrase_df[paraphrase_df["uuid"] == uuid].reset_index(drop=True)
    baselineacc = calculate_accuracy(baseline, is_multichoice=False, birdirect=True, use_generation=True, by_probs=False, label="Baseline single run", verbose=False)
    num_baseline_match = int(baselineacc * len(baseline))
    paraphraseacc = calculate_accuracy(paraphrase, is_multichoice=False, birdirect=True, use_generation=True, by_probs=False, label="paraphrase single run", verbose=False)
    num_paraphrase_match = int(paraphraseacc * len(paraphrase))
    print(f"UUID: {uuid}, Baseline correct: {num_baseline_match}/{len(baseline)}, Paraphrase correct: {num_paraphrase_match}/{len(paraphrase)}")

UUID: 5a7a06935542990198eaf050, Baseline correct: 2/4, Paraphrase correct: 4/4
UUID: 5a879ab05542996e4f30887e, Baseline correct: 3/4, Paraphrase correct: 2/4
UUID: 5a8d7341554299441c6b9fe5, Baseline correct: 0/3, Paraphrase correct: 0/4
UUID: 5a82171f5542990a1d231f4a, Baseline correct: 1/4, Paraphrase correct: 1/4
UUID: 5a84dd955542997b5ce3ff79, Baseline correct: 1/4, Paraphrase correct: 1/4
UUID: 5a7e36045542991319bc9440, Baseline correct: 0/4, Paraphrase correct: 1/4
UUID: 5adf44985542993a75d2646d, Baseline correct: 0/4, Paraphrase correct: 0/4
UUID: 5a832c3455429954d2e2ec41, Baseline correct: 0/4, Paraphrase correct: 1/4
UUID: 5a7d0db955429909bec76924, Baseline correct: 0/4, Paraphrase correct: 2/4
UUID: 5a89372855429951533612e6, Baseline correct: 0/4, Paraphrase correct: 0/4
UUID: 5abd90545542996e802b47d7, Baseline correct: 0/4, Paraphrase correct: 0/4
UUID: 5aba66c855429939ce03dcdb, Baseline correct: 0/4, Paraphrase correct: 0/4
UUID: 5a7722d655429966f1a36c99, Baseline correct: 0/

In [32]:
# for uuid in uuids:
uuid = uuids[3]
baseline = baseline_df[baseline_df["uuid"] == uuid].reset_index(drop=True)
paraphrase = paraphrase_df[paraphrase_df["uuid"] == uuid].reset_index(drop=True)

# for prompt, thinking in zip(baseline['prompt'].tolist(), baseline['thinking'].tolist()):
#     print("=====" * 8)
#     print("----- Prompt -----")
#     print(prompt)
#     print("----- Thinking -----")
#     print(thinking)

for prompt, thinking in zip(paraphrase['prompt'].tolist(), paraphrase['thinking'].tolist()):
    print("=====" * 8)
    print("----- Prompt -----")
    print(prompt)
    print("----- Thinking -----")
    print(thinking)

----- Prompt -----
What nationality was James Henry Miller's wife?
----- Thinking -----
<think>
Okay, so I need to find out the nationality of James Henry Miller's wife. Let me start by recalling who James Henry Miller is. I think he's a historical figure, maybe from the 18th or 19th century. Wait, there's a James Henry Miller who was involved in the American Revolution? Or maybe he's a different person.

First, I should confirm who James Henry Miller is. Let me check my memory. There was a James Henry Miller who was a British soldier and later a Loyalist during the American Revolution. He was part of the British forces and fought against the American colonists. After the war, he moved to Canada, specifically Nova Scotia. 

Now, his wife. If he was a British Loyalist, he might have married someone from Britain or perhaps a local woman in Nova Scotia. But I need to find specific information. I remember that he married a woman named Mary Ann. Wait, was her name Ann? Maybe Mary Ann or Ann

### Role: 
You are a Logic-Based Linguistic Architect.

### Task:
Radically re-engineer the provided Target Question. 
Your goal is to move as far away from the original wording as possible while ensuring the "Ground Truth" answer remains the only logical conclusion.

### Strategies for Audacity
- The "Hypothetical Case" (If/Suppose): Reframe the question as a scenario (e.g., "If an individual was the CEO of X, what city would they call their headquarters?").
- Entity Abstraction: Instead of naming an entity, describe its unique properties (e.g., instead of "The 44th President," use "The person who succeeded George W. Bush").
- Relational Inversion: If the question asks "Who directed X?", ask "Identify the creator whose filmography includes the title X."
- Logical Predicates: Break the question into its requirements and rebuild it from the end to the beginning.

### Constraints:
- Logical Anchoring: The answer must be 100% reachable using the same evidence used for the original.
- Least Keyword Overlap: Attempt to use 0% of the non-essential nouns/verbs from the original besides the key entities.



In [37]:
instruction = """### Role: 
You are a Logic-Based Linguistic Architect.

### Task:
Radically re-engineer the provided Target Question. 
Your goal is to move as far away from the original wording as possible while ensuring the "Ground Truth" answer remains the only logical conclusion.

### Strategies for Audacity
- The "Hypothetical Case" (If/Suppose): Reframe the question as a scenario (e.g., "If an individual was the CEO of X, what city would they call their headquarters?").
- Entity Abstraction: Instead of naming an entity, describe its unique properties (e.g., instead of "The 44th President," use "The person who succeeded George W. Bush").
- Relational Inversion: If the question asks "Who directed X?", ask "Identify the creator whose filmography includes the title X."
- Logical Predicates: Break the question into its requirements and rebuild it from the end to the beginning.

### Constraints:
- Logical Anchoring: The answer must be 100% reachable using the same evidence used for the original.
- Least Keyword Overlap: Attempt to use 0% of the non-essential nouns/verbs from the original besides the key entities.
- Output JSON schema as below:
    {
        "paraphrase": string,
    }
"""
import json
escaped_text = json.dumps({"system": instruction, "user": "### Target Question:\n{question}"}, ensure_ascii=False, indent=4)
print(escaped_text)

{
    "system": "### Role: \nYou are a Logic-Based Linguistic Architect.\n\n### Task:\nRadically re-engineer the provided Target Question. \nYour goal is to move as far away from the original wording as possible while ensuring the \"Ground Truth\" answer remains the only logical conclusion.\n\n### Strategies for Audacity\n- The \"Hypothetical Case\" (If/Suppose): Reframe the question as a scenario (e.g., \"If an individual was the CEO of X, what city would they call their headquarters?\").\n- Entity Abstraction: Instead of naming an entity, describe its unique properties (e.g., instead of \"The 44th President,\" use \"The person who succeeded George W. Bush\").\n- Relational Inversion: If the question asks \"Who directed X?\", ask \"Identify the creator whose filmography includes the title X.\"\n- Logical Predicates: Break the question into its requirements and rebuild it from the end to the beginning.\n\n### Constraints:\n- Logical Anchoring: The answer must be 100% reachable using th